# 01 — Data Cleaning

**Goal:** Load the raw NGX All-Share Index daily price data, set a proper datetime index,
handle non-trading days/gaps, and compute daily returns — the foundation every later notebook
builds on.

Output: `data/ngx_prices_clean.csv`

## 1. Load the Data

In [1]:
import pandas as pd
import numpy as np

# Expected columns from most NGX/investing.com exports: Date, Open, High, Low, Close, Volume
# Adjust column names below to match your actual downloaded file if they differ.
df = pd.read_csv("../data/ngx_all_share_index.csv")
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
print(df.columns.tolist())
df.head()

['date', 'price', 'open', 'high', 'low', 'vol.', 'change_%']


,date,price,open,high,low,vol.,change_%
0,09/15/2026,"244,442.17","243,299.23","244,541.81","243,311.13",260.67M,0.47%
1,09/14/2026,"243,299.23","243,052.73","243,370.88","242,575.20",428.94M,0.10%
2,09/11/2026,"243,052.73","242,574.48","243,052.73","242,333.17",552.87M,0.28%
3,09/10/2026,"242,378.13","242,223.09","242,483.00","241,049.27",1.40B,0.06%
4,09/09/2026,"242,223.09","244,802.11","244,823.44","241,983.66",534.34M,-1.05%


In [2]:
# Rename to match what the rest of the notebook expects
df = df.rename(columns={"price": "close", "vol.": "volume"})

# investing.com exports numbers as text with commas (e.g. "45,231.50") and
# volume with letters like "1.2M" - clean these into proper numeric types
for col in ["close", "open", "high", "low"]:
    df[col] = df[col].astype(str).str.replace(",", "").astype(float)

# change_% comes as text like "1.25%" - strip the % sign if you want to use it later
df["change_%"] = df["change_%"].astype(str).str.replace("%", "").astype(float)

df.dtypes

date            str
close       float64
open        float64
high        float64
low         float64
volume          str
change_%    float64
dtype: object

## 2. Parse Dates and Set Index

In [3]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df = df.set_index("date")

print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"Total rows: {len(df):,}")

Date range: 2012-01-30 to 2026-09-15
Total rows: 3,620


## 3. Handle Missing Trading Days

Stock exchanges don't trade every calendar day (weekends, public holidays). Rather than
forward-filling every gap silently, check how large the gaps actually are first — large gaps
might indicate a data quality issue rather than just a holiday.

In [4]:
date_diffs = df.index.to_series().diff().dt.days
print("Distribution of gaps between consecutive trading days:")
print(date_diffs.value_counts().sort_index())

large_gaps = date_diffs[date_diffs > 5]
print(f"\nGaps larger than 5 days: {len(large_gaps)}")
if len(large_gaps) > 0:
    print(large_gaps)

Distribution of gaps between consecutive trading days:
date
1.0    2806
2.0      36
3.0     691
4.0      40
5.0      45
6.0       1
Name: count, dtype: int64

Gaps larger than 5 days: 1
date
2015-12-29    6.0
Name: date, dtype: float64


**Note:** Gaps of 1-3 days are normal (weekends/single holidays). Investigate any gap larger than ~5-7 days manually — it may be a market closure (worth keeping and noting) or a genuine data export issue (worth filling or flagging).

## 4. Check for Missing Values and Duplicate Dates

In [5]:
print("Missing values per column:")
print(df.isnull().sum())

print(f"\nDuplicate dates: {df.index.duplicated().sum()}")
df = df[~df.index.duplicated(keep="first")]

Missing values per column:
close          0
open           0
high           0
low            0
volume      1297
change_%       0
dtype: int64

Duplicate dates: 0


In [6]:
print(df.columns.tolist())

['close', 'open', 'high', 'low', 'volume', 'change_%']


## 5. Compute Daily Returns

In [7]:
df["daily_return"] = df["close"].pct_change()

# drop the first row - no return can be computed for it
df = df.dropna(subset=["daily_return"])

df[["close", "daily_return"]].head()

,close,daily_return
date,,
2012-01-31,20875.83,0.006951
2012-02-01,20790.88,-0.004069
2012-02-02,20822.00,0.001497
2012-02-03,20877.64,0.002672
2012-02-07,20790.88,-0.004156


## 6. Save Cleaned Data

In [8]:
df.to_csv("../data/ngx_prices_clean.csv")
print(f"Saved {len(df):,} rows to ../data/ngx_prices_clean.csv")

Saved 3,619 rows to ../data/ngx_prices_clean.csv
